# 3.0.1 - Xử lý làm sạch dữ liệu: Loại bỏ các dòng chứa giá trị khuyết thiếu (Drop Missing Values)

### 🎯 Mục tiêu:
- Loại bỏ toàn bộ các dòng bị thiếu 1 hoặc nhiều ô dữ liệu (`NaN`) để tập dữ liệu hoàn toàn sạch, đồng nhất và sẵn sàng cho các mô hình Machine Learning / Deep Learning.

### 🔍 Phân tích nguyên nhân các giá trị NaN:
1. **59 dòng đầu tiên (1990-01-02 đến 1990-03-26):** Thiếu do các đặc trưng cửa sổ trượt dài nhất là 60 ngày (`Momentum_60D`, `Volatility_60D`) cần 60 phiên lịch sử để tích lũy giá trị.
2. **120 dòng cuối cùng (2026-04-08 đến 2026-09-28):** Thiếu do biến mục tiêu xa nhất là `Target_120` (dự báo 120 phiên tương lai) chưa có dữ liệu giá tương lai.
- **Tổng số dòng bị loại bỏ:** $59 + 120 = 179$ dòng.
- **Số dòng hoàn chỉnh còn lại:** $9,252 - 179 = 9,073$ dòng (100% không còn ô NaN nào).

---
- **Dữ liệu đầu vào:** `dataset/3.0.0_regime.csv`
- **Dữ liệu đầu ra:** `dataset/3.0.1_drop_missing.csv`

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Đọc dữ liệu từ phiên bản 3.0.0_regime.csv
dataset_dir = 'dataset' if os.path.exists('dataset') else ('../dataset' if os.path.exists('../dataset') else '.')
input_path = os.path.join(dataset_dir, '3.0.0_regime.csv')
print(f"Đọc dữ liệu từ: {input_path}")
df = pd.read_csv(input_path)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Kích thước ban đầu: {df.shape[0]} dòng, {df.shape[1]} cột.")
display(df.head(3))

In [ ]:
# 2. Thống kê chi tiết các ô bị khuyết thiếu (NaN) trên từng cột
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]

print("=== DANH SÁCH CÁC CỘT CÓ GIÁ TRỊ KHUYẾT THIẾU ===")
display(null_cols)

rows_with_null = df.isnull().any(axis=1).sum()
print(f"Tổng số dòng bị thiếu ít nhất 1 ô dữ liệu: {rows_with_null} dòng ({rows_with_null / len(df) * 100:.2f}%).")

In [ ]:
# 3. Tiến hành loại bỏ các dòng có chứa giá trị NaN
df_cleaned = df.dropna().reset_index(drop=True)

print(f"Kích thước dữ liệu sau khi dropna(): {df_cleaned.shape[0]} dòng, {df_cleaned.shape[1]} cột.")
print(f"Số lượng ô khuyết thiếu còn lại: {df_cleaned.isnull().sum().sum()}")
print(f"Khoảng thời gian dữ liệu sạch: từ {df_cleaned['Date'].iloc[0].date()} đến {df_cleaned['Date'].iloc[-1].date()}.")

print("\n=== PHÂN BỐ VOLATILITY_REGIME TRÊN TẬP DỮ LIỆU SẠCH ===")
display(df_cleaned['Volatility_Regime'].value_counts())

In [ ]:
# 4. Xuất file kết quả phiên bản 3.0.1_drop_missing.csv
output_path = os.path.join(dataset_dir, '3.0.1_drop_missing.csv')
df_cleaned.to_csv(output_path, index=False)

print(f"✅ ĐÃ XUẤT THÀNH CÔNG TẠI: {output_path}")
print(f"Kích thước Dataset mới: {df_cleaned.shape[0]} dòng, {df_cleaned.shape[1]} cột (100% sạch, không còn NaN).")
display(df_cleaned.head(3))
display(df_cleaned.tail(3))